**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [1]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

profiles = dh.get_k8s_resource_profiles()
print(profiles)

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

['1xrtxa5000', '1xv100', 'cpu', '1xa40-shared', '2xa40-shared', '3xa40-shared', '4xa40-shared', '5xa40-shared', '6xa40-shared', '7xa40-shared', '8xa40-shared', '1xrtx5000-shared', '2xrtx5000-shared', '1xrtxa5000-shared', '2xrtxa5000-shared', '3xrtxa5000-shared', '1xv100-shared', '2xv100-shared', '3xv100-shared', '4xv100-shared', '5xv100-shared', '6xv100-shared', '7xv100-shared', '8xv100-shared', 'cpu-shared', 'default']
Progetto: floods


**SETUP PARAMETERS**

In [8]:
# Parametri Job   
job_name = "visual_opt_2D_convLstm_v3"                                
dataset = "Test" 
test_sar = False
test_opt = True                                       
#handler = pretrain_encoders                                         

parametri = {     
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                      
    "n_images2": 4, "n_channels2": 10,  
    "output_dim": 16,                              
    "mamba": False, 
    "dataset": dataset,
    "test_sar": test_sar,                                 
    "test_opt": test_opt,
    "weights_s1": "encoder-s1-weights_train_sar_2D_convLstm_v1_Standard_200",
    "weights_s2": "encoder-s2-weights_train_opt_2D_convLstm_v1_Standard_200",
    "job_name": job_name,
    "ltae": False,
    "monodimensional": False,
    "n_samples": 10,         
    "recon_channels": [2,1,0]
}                                                              

print(f"PARAMETRI: {parametri}")

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

PARAMETRI: {'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'output_dim': 16, 'mamba': False, 'dataset': 'Test', 'test_sar': False, 'test_opt': True, 'weights_s1': 'encoder-s1-weights_train_sar_2D_convLstm_v1_Standard_200', 'weights_s2': 'encoder-s2-weights_train_opt_2D_convLstm_v1_Standard_200', 'job_name': 'visual_opt_2D_convLstm_v3', 'ltae': False, 'monodimensional': False, 'n_samples': 10, 'recon_channels': [2, 1, 0]}


**BUILD ENVIRONMENT**

In [9]:
test_train_func = project.new_function(
    name= f'encoders-Floods_{dataset}_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="test_encoders_visual", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "torch==2.1.2", "matplotlib==3.10.9", "digitalhub==0.15.11", "digitalhub-runtime-python==0.15.2"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = test_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

2026-09-24 09:19:54,737 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run feb2e405cdac414a916a1a3c0c8817d0 to finish...
2026-09-24 09:19:59,742 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run feb2e405cdac414a916a1a3c0c8817d0 to finish...
2026-09-24 09:20:04,750 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run feb2e405cdac414a916a1a3c0c8817d0 to finish...
2026-09-24 09:20:09,760 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run feb2e405cdac414a916a1a3c0c8817d0 to finish...
2026-09-24 09:20:14,872 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run feb2e405cdac414a916a1a3c0c8817d0 to finish...
2026-09-24 09:20:19,879 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run feb2e405cdac414a916a1a3c0c8817d0 to finish...
2026-09-24 09:20:24,887 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run feb2e405cdac414a916a1a3c0c8817d0 to finish...
2026-09-24 09

BUILD: COMPLETED


**TRAINING**

In [10]:
# action job = avvia container, esegue script, libera risorse

run_test_encoders = test_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xv100-shared",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run test_encoders avviato: {run_test_encoders.id}")
print(run_test_encoders.status.state)
print(run_test_encoders.status.message)

2026-09-24 09:20:50,102 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 5718ebc782dc452d89b29d5acb9c2f87 to finish...
2026-09-24 09:20:55,108 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 5718ebc782dc452d89b29d5acb9c2f87 to finish...
2026-09-24 09:21:00,116 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 5718ebc782dc452d89b29d5acb9c2f87 to finish...
2026-09-24 09:21:05,125 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 5718ebc782dc452d89b29d5acb9c2f87 to finish...
2026-09-24 09:21:10,134 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 5718ebc782dc452d89b29d5acb9c2f87 to finish...
2026-09-24 09:21:15,142 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 5718ebc782dc452d89b29d5acb9c2f87 to finish...
2026-09-24 09:21:20,254 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 5718ebc782dc452d89b29d5acb9c2f87 to finish...
2026-09-24 09

Run test_encoders avviato: 5718ebc782dc452d89b29d5acb9c2f87
COMPLETED
None
